In [8]:
!pip install pandas sqlalchemy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
import pandas as pd
from sqlalchemy import create_engine

# Raw data with string inconsistencies, mixed units, and formatting errors
raw_mining_data = {
    'sample_id': ['S-001', 'sample_02', 'S.003', 'S-004', 'SAMPLE_05', 's006', 'S-007', 'S-008'],
    'ore_grade': ['0.55%', '0.12%', '0.88%', '450ppm', '1.20%', '0.05%', 'below_detection', '15.5%'],
    'site_zone': ['north-zone', 'NORTH_ZONE', 'North', 'south', 'South_Zone', 'south', 'NORTH', 'South'],
    'extraction_date': ['2023-01-10', '10/01/2023', '2023.01.11', '2023-01-12', '12-01-2023', '2023-01-13', '2023-01-14', '2023-01-15']
}

df_raw = pd.DataFrame(raw_mining_data)
engine = create_engine('sqlite://')
df_raw.to_sql('raw_ore_samples', engine, index=False)

print("Database 'raw_ore_samples' created and ready for SQL cleaning.")
print(df_raw.head(8))

Database 'raw_ore_samples' created and ready for SQL cleaning.
   sample_id        ore_grade   site_zone extraction_date
0      S-001            0.55%  north-zone      2023-01-10
1  sample_02            0.12%  NORTH_ZONE      10/01/2023
2      S.003            0.88%       North      2023.01.11
3      S-004           450ppm       south      2023-01-12
4  SAMPLE_05            1.20%  South_Zone      12-01-2023
5       s006            0.05%       south      2023-01-13
6      S-007  below_detection       NORTH      2023-01-14
7      S-008            15.5%       South      2023-01-15


In [13]:
# Query di anteprima - NON modifica tabella l'originale
q = """
SELECT 
    CASE
        WHEN sample_id = 'sample_02' THEN 'S-002'
        WHEN sample_id = 'S.003' THEN 'S-003'
        WHEN sample_id = 'SAMPLE_05' THEN 'S-005'
        WHEN sample_id = 's006' THEN 'S-006'
        ELSE sample_id
    END AS sample_id,
    
    CASE
        WHEN site_zone = 'north-zone' THEN 'North'
        WHEN site_zone = 'NORTH_ZONE' THEN 'North'
        WHEN site_zone = 'NORTH' THEN 'North'
        WHEN site_zone = 'south' THEN 'South'
        WHEN site_zone = 'South_Zone' THEN 'South'
        ELSE site_zone
    END AS site_zone,
    
    ore_grade,
    extraction_date
FROM raw_ore_samples;
"""

df_finale = pd.read_sql_query(q, engine)
print("Tutti i dati puliti:")
print(df_finale)

Tutti i dati puliti:
  sample_id site_zone        ore_grade extraction_date
0     S-001     North            0.55%      2023-01-10
1     S-002     North            0.12%      10/01/2023
2     S-003     North            0.88%      2023.01.11
3     S-004     South           450ppm      2023-01-12
4     S-005     South            1.20%      12-01-2023
5     S-006     South            0.05%      2023-01-13
6     S-007     North  below_detection      2023-01-14
7     S-008     South            15.5%      2023-01-15
